# Практикум 04 · Улучшение изображения

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 04. Шум, яркость и восстановление повреждения

**Работа на нашем роботе:** создайте [личную копию курса и выберите нужное окружение](../docs/SETUP_ROBOT.md). Все команды ниже выполняйте из её корня; результаты `outputs/` будут принадлежать вашей копии. Самостоятельный поиск камеры, контейнера и ROS-топиков разобран в той же инструкции.

**Тема Yahboom:** 9.4 Image beautification. **Время:** 60–90 минут.

## Цель и место выполнения

Сравните сглаживание, изменение яркости, усиление локального контраста и заполнение небольшого повреждения. Оценивайте, как операция влияет на полезные признаки: края предмета, цветовую маску и читаемость мелких деталей. Красивое изображение и удобное изображение для следующего алгоритма не всегда совпадают.

Раздел производителя включает восстановление, увеличение яркости и сглаживание. Здесь эти темы объединены в воспроизводимый эксперимент на собственной синтетической сцене. Работайте на ноутбуке или компьютере неподвижного ROSMASTER из корня курса. Нужны `data/scene.png` и `data/noisy.png`; их создаёт `python3 tools/generate_data.py`.

## Пять понятий

**Шум** — нежелательные изменения пикселей. В учебном файле к сцене добавлены случайные отклонения и отдельные чёрные/белые точки. Генератор использует фиксированное семя, поэтому одни и те же настройки дают повторяемый эксперимент. На настоящей камере шум и причины его появления будут иными.

**Сглаживание** объединяет информацию соседей. Гауссов фильтр усредняет с весами, медианный выбирает медианное значение в окрестности. Двусторонний, или bilateral, учитывает и расстояние между пикселями, и различие их яркостей/цветов. Увеличение окна делает обработку сильнее, но может ухудшить мелкие детали.

**Линейная коррекция** вычисляет `новое = alpha × исходное + beta`. Результат ограничивается диапазоном 0–255. `alpha` меняет масштаб значений, `beta` добавляет постоянную величину. Слишком большое увеличение приводит к насыщению: разные светлые значения превращаются в одинаковый белый.

**CLAHE** изменяет локальный контраст с ограничением усиления. В программе он применяется к светлоте L пространства Lab, после чего изображение возвращается в BGR. Метод может показать детали в малоразличимых участках, но способен подчеркнуть и шум.

**Восстановление, или inpainting**, заполняет отмеченную область по её окружению. Маска показывает, где данные повреждены. Алгоритм не знает, что действительно находилось за царапиной: результат является оценкой. Для измерений нельзя выдавать восстановленные пиксели за исходные наблюдения камеры.

## Практика: семь шагов

1. Посмотрите параметры и обработайте зашумлённую сцену:

   ```bash
   python3 -m rosmaster_cv.lesson04_enhancement --help
   python3 -m rosmaster_cv.lesson04_enhancement --input data/noisy.png --reference data/scene.png --out outputs/04
   ```

2. Откройте `comparison.png`. В верхнем ряду расположены исходное зашумлённое изображение, Gaussian и median; в нижнем — bilateral, CLAHE и изменение яркости. Для проверки мелких деталей открывайте отдельные PNG в исходном размере: сравнительный лист уменьшает изображения.

3. Откройте `report.json`. Поле `mean_gray` показывает среднюю яркость, а `mse_to_reference` — средний квадрат ошибки относительно чистой сцены. Меньшая MSE означает более близкие пиксели именно к этому эталону. Она не доказывает, что красный предмет будет находиться лучше на любой фотографии.

4. Увеличьте окно сглаживания:

   ```bash
   python3 -m rosmaster_cv.lesson04_enhancement --kernel 9 --reference data/scene.png --out outputs/04_k9
   ```

   Сравните точки шума и края текста в `median.png` и `gaussian.png` для окон 5 и 9. Запишите, какой вариант лучше сохраняет мелкие детали, и приведите конкретный участок изображения.

5. Измените яркость в обе стороны:

   ```bash
   python3 -m rosmaster_cv.lesson04_enhancement --alpha 1.3 --beta 50 --out outputs/04_light
   python3 -m rosmaster_cv.lesson04_enhancement --alpha 1.0 --beta -40 --out outputs/04_dark
   ```

   Откройте именно `brightness.png`. Найдите на шкале яркости участки, где соседние ступени перестали различаться. Сопоставьте наблюдение со значениями `alpha` и `beta`.

6. Исследуйте искусственную царапину. Стандартный запуск создаёт `damaged.png`, `damage_mask.png` и `inpainted.png`. В маске белым отмечена узкая вертикальная полоса. Сравните восстановление на ровном фоне и на границе цветной фигуры. Затем обработайте сохранённое повреждение с явной маской:

   ```bash
   python3 -m rosmaster_cv.lesson04_enhancement --input outputs/04/damaged.png --mask outputs/04/damage_mask.png --inpaint-radius 5 --out outputs/04_restore
   ```

   При `--mask` программа считает вход уже повреждённым и новую царапину не добавляет.

7. Сравните два уровня CLAHE:

   ```bash
   python3 -m rosmaster_cv.lesson04_enhancement --clip-limit 8 --out outputs/04_clahe8
   ```

   Откройте `clahe.png` для стандартного ограничения 2 и нового 8. Укажите, какие детали стали заметнее и где усиление мешает.

## Ожидаемые файлы

В каждом запуске создаются `noisy.png` (копия входа), `gaussian.png`, `median.png`, `bilateral.png`, `clahe.png`, `brightness.png`, `damaged.png`, `damage_mask.png`, `inpainted.png`, `comparison.png` и `report.json`. Название `noisy.png` используется и для собственной фотографии: программа не утверждает, что каждый вход обязательно содержит искусственный шум.

## Самостоятельное изменение

Выберите два варианта обработки и проверьте на них цветовую маску третьего урока с одинаковыми порогами. Например, подайте `outputs/04/median.png` и `outputs/04/brightness.png` через `--input` в `lesson03_drawing`, задав разные папки `--out`. Сравните пропуски объекта и лишние белые участки. Опишите, какая обработка помогает именно вашему детектору и при каких параметрах. Добавьте собственный снимок в слабом освещении; для него не используйте синтетическую сцену как эталон MSE.

## Три вопроса

1. Почему сильное сглаживание может одновременно убрать шум и испортить полезный контур?
2. Что происходит со значением 240 при `alpha=1` и `beta=30`?
3. Почему восстановленный участок нельзя считать достоверным наблюдением?

## Что сдать и ошибки

Сдайте сравнительный лист, таблицу двух окон, результаты восстановления и две маски после разных обработок. Объясните выбор параметров и ограничения MSE.

**Маска не подходит:** её ширина и высота должны совпадать с повреждённым входом; белым отмечают только повреждение. **Неверное окно:** используйте нечётный `--kernel` от 3 до 31.

## Источник и далее

[Yahboom 9.4](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8145/). Сравнение фильтров, CLAHE и численная оценка добавлены для самостоятельного эксперимента.

Далее: [05 — Практические приложения](../lessons/05_applications.md).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
run_tool("generate_data.py")
run_module("lesson04_enhancement", "--input", "data/noisy.png", "--out", "outputs/04")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results("outputs/04")


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
